In [ ]:
#Importing libraries
import pandas as pd 
import numpy as np
import pickle 
import os
from datetime import datetime

from tqdm import tqdm     #progress bar for pandas operations
tqdm.pandas()

print("Libraries loaded.")
print(f"Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

: 

In [ ]:
#Getting the project root path 
import os 
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data")
RAW_DATA_DIR = os.path.join(DATA_DIR, "raw/ml-latest")
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, "processed")
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

print(f"Base directory: {BASE_DIR}")


Base directory: c:\Projects\Cinemate V2


In [ ]:
#Loading the data 
movies_data = pd.read_csv(os.path.join(RAW_DATA_DIR, "movies.csv"))
rating_data = pd.read_csv(os.path.join(RAW_DATA_DIR, "ratings.csv"))
link_data = pd.read_csv(os.path.join(RAW_DATA_DIR, "links.csv"))
tag_data= pd.read_csv(os.path.join(RAW_DATA_DIR, "tags.csv"))
genome_tags = pd.read_csv(os.path.join(RAW_DATA_DIR, "genome-tags.csv"))
genome_scores = pd.read_csv(os.path.join(RAW_DATA_DIR, "genome-scores.csv"))


In [ ]:
#Creating a dict of datasets for easy processing
dataset_dict = {
    "movies": movies_data,
    "ratings": rating_data,
    "links": link_data,
    "tags": tag_data,
    "genome_tags": genome_tags,
    "genome_scores": genome_scores
}

### 1. Removing Null values

**Removing Null values**

- From EDA , it is known that there are 17 null values in tag data and 126 null values in link data, tmdbId column.
- As the number 17 is quite low when compared with number of data point tags data holding which is ``2328315`` so its better to drop it rather than using mode.
- for tmdbId this is the column which can be used to connect with external websites , so it is a good option to do imputation over here .

In [ ]:
#Removing null values from tag data
tag_data.dropna(inplace=True)
tag_data.isnull().sum()

userId       0
movieId      0
tag          0
timestamp    0
dtype: int64

In [ ]:
#Removing null values from link data
link_data.fillna(0, inplace=True)
link_data.isnull().sum()

movieId    0
imdbId     0
tmdbId     0
dtype: int64

### 2. Memory Optimisation

**the dataset is consuming about 1.69gb of RAM which is high. to reduce the memory consumption figure we can reduce the size of the each value from float64/int64 to float32/int32 or float16/int16**


As the memory usage is quite high (Total memory usage of all datasets: 1,775,236,734 bytes ≈ 1692.99 MB), there is a possibility of system slowdown or failure during collaborative filtering (CF) model training. Therefore, it is better to reduce the size of each value by converting the data type from float64 to float32 or float16.


but this step need to be done in a way such that no data must be lost , by observing data closly we can say this memory conversion can work
- userId (total users 3.3 lakhs) --> int32 
- movieId (total movies 2.8 lakhs) --> int32
- rating (fractional bounded value in range of 0.5 to 5) --> float16
- timestamp (growing data, with max value 170 crore)  --->int32 (Suitable for static data)
- tmdbId int32
- imdbId int32
- tagId float16
- relevance float32
 
and rest of column no memory conversion

In [ ]:
def memory_used(df : pd.DataFrame) -> float:
    """Calculate memory used by a DataFrame in MB."""
    return df.memory_usage(deep=True).sum()/1024**2

def compare_memory_usage(df1: pd.DataFrame, df2: pd.DataFrame, dataframe_name: str):
    print(f"Memory usage of {dataframe_name} : {memory_used(df1):.2f} MB")
    print(f"Memory usage of {dataframe_name} after conversion: {memory_used(df2):.2f} MB")
    

In [ ]:
#creating a dictionary to store the column name and the data type to convert to
dtype_dict= {
    "userId": "int32",
    "movieId": "int32",
    "rating": "float16",
    "timestamp": "int32",
    "tmdbId": "int32",
    "imdbId": "int32",
    "tagId": "int32",
    "relevance": "float32"
}

In [ ]:
def moemory_conversion(df_name :str, df: pd.DataFrame,):
    before = memory_used(df)
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype('category')
            continue
        if col in dtype_dict:
            df[col] = df[col].astype(dtype_dict[col])
        
    after = memory_used(df)

    print(f"Memory usage before conversion: {before:.2f} MB")
    print(f"Memory usage after conversion: {after:.2f} MB")
    

In [ ]:
#Applying memory optimization to all datasets
for name, df in dataset_dict.items():
    print("="*75)
    print(f"\nOptimizing memory for {name} dataset:")
    moemory_conversion(name, df)
    print("\n")
    print("="*75)


Optimizing memory for movies dataset:
Memory usage before conversion: 13.33 MB
Memory usage after conversion: 10.03 MB



Optimizing memory for ratings dataset:
Memory usage before conversion: 1032.48 MB
Memory usage after conversion: 451.71 MB



Optimizing memory for links dataset:
Memory usage before conversion: 1.98 MB
Memory usage after conversion: 0.99 MB



Optimizing memory for tags dataset:
Memory usage before conversion: 222.33 MB
Memory usage after conversion: 68.03 MB



Optimizing memory for genome_tags dataset:
Memory usage before conversion: 0.08 MB
Memory usage after conversion: 0.11 MB



Optimizing memory for genome_scores dataset:
Memory usage before conversion: 422.79 MB
Memory usage after conversion: 211.40 MB




In [ ]:
#Total memory usage
total_memory_usage = 0
for dataset_name, dataset in dataset_dict.items():
    total_memory_usage+= dataset.memory_usage(deep=True).sum()

#print total memory usage of all datasets
print(f"Total memory usage of all datasets: {total_memory_usage} bytes")
#in megabytes
print(f"Total memory usage of all datasets: {total_memory_usage/1024/1024} MB")


# Percentage reduction in memory usage
initial_memory_usage = 1693.1785593032837
reduction_percentage = ((initial_memory_usage - total_memory_usage/1024/1024) / initial_memory_usage) * 100
print(f"Percentage reduction in memory usage: {reduction_percentage:.4f}%")


Total memory usage of all datasets: 778325305 bytes
Total memory usage of all datasets: 742.268853187561 MB
Percentage reduction in memory usage: 56.1612%


### 3. filtering out low activity user and movies

In [ ]:
#filtering
print("Before filtering:")
print(f"Number of users : {rating_data['userId'].nunique()}")
print(f"Number of movies : {rating_data['movieId'].nunique()}")
print(f"Ratings: {len(rating_data)}")

#Filter users
user_counts = rating_data.groupby("userId").size()
valid_user = user_counts[user_counts>=20].index   #return valid users id
rating_data = rating_data[rating_data['userId'].isin(valid_user)]
print(f"After user filter : {rating_data['userId'].nunique()} users remaining.")


#filter movies
movies_count = rating_data.groupby("movieId").size()
valid_movies = movies_count[movies_count>=10].index
rating_data = rating_data[rating_data['movieId'].isin(valid_movies)]
print(f"After movie filtering : {rating_data['movieId'].nunique()} movies remaining")


print(f"Ratings remaining : {len(rating_data)}")
print(f"Retained :{100*len(rating_data)/33832162:.1f}% of original")


Before filtering:
Number of users : 330975
Number of movies : 83239
Ratings: 33832162
After user filter : 204443 users remaining.
After movie filtering : 31837 movies remaining
Ratings remaining : 32335389
Retained :95.6% of original


### 4. Time based splitting

In [ ]:
#Sorting on the basis of timestamp 
rating_data= rating_data.sort_values('timestamp').reset_index(drop=True)

#verify sorting
assert rating_data['timestamp'].is_monotonic_increasing, \
    "Sort failed — timestamps not monotonically increasing!"
print("Sort verified — timestamps are monotonically increasing")
print()

#show data range
earliest = pd.to_datetime(rating_data.timestamp.iloc[0], unit='s')
newest = pd.to_datetime(rating_data.timestamp.iloc[-1], unit="s")

#splitting
splitting_val = int(len(rating_data) * 0.8)
train =rating_data.iloc[:splitting_val].copy()
test = rating_data.iloc[splitting_val:].copy()

split_date= pd.to_datetime(rating_data.iloc[splitting_val]['timestamp'], unit='s')

print(f"Splitting index : {splitting_val}")
print(f"Split date: {split_date}")
print(f"Train size : {train.shape} \n"
      f"Test size: {test.shape}.")

# Leakage check
assert train['timestamp'].max() <= test['timestamp'].min(), \
    "DATA LEAKAGE — timestamps overlap between train and test!"
print("PASS — No timestamp leakage confirmed")


Sort verified — timestamps are monotonically increasing

Splitting index : 25868311
Split date: 2018-07-22 06:49:46
Train size : (25868311, 4) 
Test size: (6467078, 4).
PASS — No timestamp leakage confirmed


### 5. Cold start analysis 

- Cold users -> User who appeared in test but not in train so user will be not seen by model during training so it cant predict what    could be the next recommendation for person (need to be removed)
- Cold movies -> same for the case of cold movies , movie which is part of test set but not part of training data. we can keep it because additional features like that of tags, genres , and titles.

In [ ]:
train_users = set(train['userId'].unique())
train_movies = set(train['movieId'].unique())
test_users = set(test['userId'].unique())
test_movies = set(test['movieId'].unique())

cold_users = test_users - train_users
cold_movies = test_movies - train_movies

print("Cold Start analysis")
print(f"Users in test not in train : {len(cold_users)} \n"
      f"({100 * len(cold_users)/len(test_users):.2f}% of test users.)")
print(f"Movies in test not in train: {len(cold_movies)} \n"
      f"{100 * len(cold_movies)/len(test_movies):.2f}% of total movies.")
print()
print("Handling strategy")
print("Handling strategy:")
print("  Cold start USERS  → remove from test (model has no embeddings)")
print("  Cold start MOVIES → kept — content tower handles via title+genres")

# Remove cold start users from test
test = test[test['userId'].isin(train_users)].copy()
print(f"\nTest after removing cold start users: {len(test):,} ratings")

Cold Start analysis
Users in test not in train : 31309 
(79.79% of test users.)
Movies in test not in train: 4071 
12.93% of total movies.

Handling strategy
Handling strategy:
  Cold start USERS  → remove from test (model has no embeddings)
  Cold start MOVIES → kept — content tower handles via title+genres

Test after removing cold start users: 1,001,865 ratings


79.79% of test users are cold start users removed from evaluation.
This is high but expected — the dataset shows heavy user growth
post-2018, meaning most recent users have no training history.

The model is evaluated on 1,001,865 ratings from ~8,000 established users.
This is a known limitation of strict time-based splitting with
fast-growing datasets.

### 6. Encoders

**Why Required?**

Raw ids are very sparse as 1 -> 3 -> 8 or so .

Embedding layers need dense 0-indexed integers.

In [ ]:
unique_users = sorted(train['userId'].unique())
unique_movies = sorted(train['movieId'].unique())

user2idx= {uid: idx for idx, uid in enumerate(unique_users)}
movie2idx = {mid: idx for idx, mid in enumerate(unique_movies)}
idx2user = {idx: uid for uid, idx in user2idx.items()}
idx2movie = {idx: mid for mid, idx in movie2idx.items()}

NUM_USERS= len(user2idx)
NUM_MOVIES = len(movie2idx)

print(f"Encoder built from train set only")
print(f"  NUM_USERS  : {NUM_USERS:,}")
print(f"  NUM_MOVIES : {NUM_MOVIES:,}")
print()

#Sample check
sample_users  = unique_users[:3]
sample_movies = unique_movies[:3]
print("sample user encoding :")
for user in sample_users:
    print(f"User id {user} --> idx {user2idx[user]}")

print("Sample movies encoding")
for movie in sample_movies:
    print(f"Movie Id {movie} --> idx {movie2idx[movie]}")


Encoder built from train set only
  NUM_USERS  : 173,134
  NUM_MOVIES : 27,766

sample user encoding :
User id 1 --> idx 0
User id 2 --> idx 1
User id 4 --> idx 2
Sample movies encoding
Movie Id 1 --> idx 0
Movie Id 2 --> idx 1
Movie Id 3 --> idx 2


In [ ]:
#Saving encoders 
ENCODER_DIR = os.path.join(PROCESSED_DATA_DIR, "encoders")
os.makedirs(ENCODER_DIR, exist_ok=True)

with open(os.path.join(ENCODER_DIR, "user2idx.pkl"), "wb") as f:
    pickle.dump(user2idx, f)
with open(os.path.join(ENCODER_DIR, "movie2idx.pkl"), "wb") as f:
    pickle.dump(movie2idx, f)
with open(os.path.join(ENCODER_DIR, "idx2user.pkl"), "wb") as f:
    pickle.dump(idx2user, f)
with open(os.path.join(ENCODER_DIR, "idx2movie.pkl"), "wb") as f:
    pickle.dump(idx2movie, f)

print(f"All encoders saved to {ENCODER_DIR}")

All encoders saved to c:\Projects\Cinemate V2\data\processed\encoders


In [ ]:
#Applying encoders
train['user_idx'] =train['userId'].map(user2idx)
train['movie_idx']= train['movieId'].map(movie2idx)

test["user_idx"] = test['userId'].map(user2idx)
test['movie_idx']= test['movieId'].map(movie2idx)

train = train.dropna(subset=['user_idx', 'movie_idx'])
test = test.dropna(subset=['user_idx', 'movie_idx'])

#Converting datatypes to int32
for col in ['movie_idx', "user_idx"]:
    train[col] = train[col].astype(np.int32)
    test[col] = test[col].astype(np.int32)

print("Encoder succesfully applied.")
print()
print("train sample : ")
print(train[['userId','movieId','rating',
             'timestamp','user_idx','movie_idx']].head(3).to_string())
print()
print("Test sample:")
print(test[['userId','movieId','rating',
            'timestamp','user_idx','movie_idx']].head(3).to_string())


Encoder succesfully applied.

train sample : 
   userId  movieId  rating  timestamp  user_idx  movie_idx
0  149954     1176     4.0  789652004     78489       1146
1   94532       21     3.0  789652009     49659         20
2   94532     1079     3.0  789652009     49659       1051

Test sample:
          userId  movieId  rating   timestamp  user_idx  movie_idx
25868311   99978     1196     3.0  1532242186     52433       1164
25868312   99978     5902     3.0  1532242189     52433       5736
25868313   99978     7147     3.0  1532242193     52433       6945


### 7. Build Genome Tag Strings

The content tower takes text input per movie.
We build one string per movie combining three sources:

- Source 1 — title       : always available (100% coverage)
- Source 2 — genres      : always available (100% coverage)
- Source 3 — genome tags : 64.2% coverage via genome-scores

Why genome tags over raw tags?
Genome tags have a relevance SCORE (0–1) per movie-tag pair.
We take only tags with relevance > 0.3 — higher quality signal
than raw user tags which can be noisy/irrelevant.

Final string format:
"{title} | {genres} | {top_genome_tags}"

Example:
"Inception (2010) | Action Sci-Fi Thriller | mind-bending
surreal thought-provoking nonlinear"

In [ ]:
genome_merged = genome_scores.merge(genome_tags, on="tagId")

#we will take referecne as 0.3 for relevance score .
HIGH_RELEVANCE = 0.3
genome_filtered = genome_merged[genome_merged['relevance']>HIGH_RELEVANCE].copy()

print(f"length of genome merged dataset : {len(genome_merged)}")
print(f"length of genome filtered dataset : {len(genome_filtered)}")

#top 10 tags per movies
genome_tag_strings = genome_filtered.sort_values(by = "relevance", ascending=False).groupby('movieId').apply(
                        lambda x: ' '.join(x['tag'].head(10).str.lower().str.strip())).reset_index().rename(columns={0: 'genome_tag_string'})

print(f"Movies with genome tag strings: {len(genome_tag_strings):,}")
print()
print(f"Sample genome tag strings : {genome_tag_strings.head(5)} ")


length of genome merged dataset : 18472128
length of genome filtered dataset : 1697802
Movies with genome tag strings: 16,376

Sample genome tag strings :    movieId                                  genome_tag_string
0        1  toys computer animation pixar animation animat...
1        2  adventure children fantasy kids childhood spec...
2        3  good sequel sequel sequels comedy original min...
3        4  women chick flick girlie movie romantic divorc...
4        5  good sequel sequel sequels midlife crisis preg... 


C:\Users\Tushar\AppData\Local\Temp\ipykernel_24364\30175304.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  genome_tag_strings = genome_filtered.sort_values(by = "relevance", ascending=False).groupby('movieId').apply(


### 8. Build User Raw Tag Strings

In [ ]:
# Also aggregate raw user tags per movie
tag_data['clean_tag']= (tag_data['tag'].astype(str).str.lower().str.strip())

raw_tag_strings = (tag_data.groupby('movieId')['clean_tag'].apply(
    lambda x: " ".join(x.unique())).reset_index().rename(columns={'clean_tag': 'raw_tag_string'}))

print(f"Movies with raw tag strings: {len(raw_tag_strings)}")
raw_tag_strings


Movies with raw tag strings: 53452


,movieId,raw_tag_string
0,1,animation friendship toys disney pixar cgi cla...
1,2,animals based on a book fantasy magic board ga...
2,3,sequel moldy old old age old men wedding old p...
3,4,characters chick flick girl movie revenge clv ...
4,5,family pregnancy wedding 4th wall aging baby d...
...,...,...
53447,288765,post-apocalyptic survival tw suicide apocalyps...
53448,288779,don camillo series
53449,288849,addiction animation short film
53450,288937,anime


### 9. Build movies_clean and Content Strings

In [ ]:
#Filter movies to those in training set only 
train_movie_ids =set(train['movieId'].unique())
movies_clean = movies_data[movies_data['movieId'].isin(train_movie_ids)].copy()

# Convert category back to string for processing
movies_clean['title']  = movies_clean['title'].astype(str)
movies_clean['genres'] = movies_clean['genres'].astype(str)

movies_clean['genres_clean'] = (movies_clean['genres'].str.replace("|", " ",
                                regex=False).str.replace('(no genres listed)', '', regex=False).str.strip())

#merge genome tags
movies_clean= movies_clean.merge(genome_tag_strings, on='movieId', how='left',)

#merge raw user tags
movies_clean = movies_clean.merge(raw_tag_strings, on='movieId', how="left")

#Fix nulls
movies_clean['genome_tag_string'] = movies_clean['genome_tag_string'].fillna("")
movies_clean['raw_tag_string']= movies_clean['raw_tag_string'].fillna("")

#Build final content strings
def build_content_string(row):
    parts = []
    parts.append(row['title'])
    if row['genres_clean']:
        parts.append(row['genres_clean'])

    #Prefer genome tags 
    if row['genome_tag_string']:
        parts.append(row['genome_tag_string'])

    elif row['raw_tag_string']:
        # fallback to raw tags if no genome tags
        parts.append(row['raw_tag_string'][:200])
    return ' | '.join(p for p in parts if p)

movies_clean['content_string'] = movies_clean.apply(build_content_string, axis=1)

#adding movie idx
movies_clean['movie_idx']= movies_clean['movieId'].map(movie2idx)
movies_clean = movies_clean.dropna(subset=['movie_idx'])
movies_clean['movie_idx'] = movies_clean['movie_idx'].astype(np.int32)


print(f"movies_clean shape: {movies_clean.shape}")
print()
for _, row in movies_clean.sample(4, random_state=42).iterrows():
    print(f"\n  [{int(row['movie_idx'])}] {row['content_string'][:90]}...")

movies_clean shape: (27766, 8)


  [17181] Charm School (Niñas mal) (2007) | Comedy | body cruelty dark comedy friend high school mak...

  [23399] Carjacked (2011) | Action Thriller | bank robbery carjacking crime kidnapping maria bello ...

  [5211] New Guy, The (2002) | Comedy | cheerleading teen movie high school teen geeks teens comedy...

  [5282] Sorry, Wrong Number (1948) | Drama Film-Noir Thriller | tense noir thriller hitchcock susp...


### 10. Content String Quality Check

In [ ]:
total = len(movies_clean)

has_genome = (movies_clean['genome_tag_string'] != "").sum()
has_raw    = ((movies_clean['genome_tag_string'] == '') &
              (movies_clean['raw_tag_string'] != '')).sum()
no_tags    = ((movies_clean['genome_tag_string'] == '') &
              (movies_clean['raw_tag_string'] == '')).sum()

print("Content String Coverage:")
print(f"  Title + Genres + Genome tags : {has_genome:,} "
      f"({100*has_genome/total:.1f}%) ← best quality")
print(f"  Title + Genres + Raw tags    : {has_raw:,} "
      f"({100*has_raw/total:.1f}%) ← fallback")
print(f"  Title + Genres only          : {no_tags:,} "
      f"({100*no_tags/total:.1f}%) ← cold start case")
print()

# Token length check — DistilBERT max is 512 tokens
word_counts = movies_clean['content_string'].str.split().str.len()
over_limit  = (word_counts > 400).sum()
print(f"String length stats:")
print(f"  Mean words   : {word_counts.mean():.0f}")
print(f"  Max words    : {word_counts.max()}")
print(f"  Over 400 words (near DistilBERT limit): {over_limit}")
print("  → These will be truncated automatically by tokenizer")


Content String Coverage:
  Title + Genres + Genome tags : 15,085 (54.3%) ← best quality
  Title + Genres + Raw tags    : 11,016 (39.7%) ← fallback
  Title + Genres only          : 1,665 (6.0%) ← cold start case

String length stats:
  Mean words   : 20
  Max words    : 59
  Over 400 words (near DistilBERT limit): 0
  → These will be truncated automatically by tokenizer


### 11. Build User Positive Sets for BPR

In [ ]:
# BPR trains on: (user, positive_movie, negative_movie)
# Positive = movies user rated >= 3.5
# Negative = sampled at training time from movies NOT in this set

POSITIVE_THRESHOLD = 3.5

train_positive = train[train['rating']>= POSITIVE_THRESHOLD].copy()
print("total length rating ", len(train))
print(f"Positive (rating >={POSITIVE_THRESHOLD}) : {len(train_positive)}    {100 * len(train_positive)/len(train):.2f}%")
print(f"Neutral or Negative rating : {len(train) - len(train_positive)}")
print()

# Build dict: user_idx → set of positive movie_idxs
user_positive_sets = (train_positive.groupby("user_idx")['movie_idx'].apply(set).to_dict())

sample_uid = list(user_positive_sets.keys())[0]
print(f"Sample — user_idx {sample_uid} has "
      f"{len(user_positive_sets[sample_uid])} positive movies")

with open(os.path.join(PROCESSED_DATA_DIR,'user_positive_sets.pkl'), 'wb') as f:
    pickle.dump(user_positive_sets, f)
print("Saved user_positive_sets.pkl")



total length rating  25868311
Positive (rating >=3.5) : 16093850    62.21%
Neutral or Negative rating : 9774461

Sample — user_idx 0 has 52 positive movies
Saved user_positive_sets.pkl


### 12. Merge Links for Streamlit Poster Display

In [ ]:
movies_clean = movies_clean.merge(link_data, on="movieId", how="left")

movies_clean['tmdbId'] = movies_clean['tmdbId'].fillna(0).astype(np.int32)
movies_clean['imdbId'] = movies_clean['imdbId'].fillna(0).astype(np.int32)

### 13. Save All Processed Files

In [ ]:
print("Saving processed files.......\n")

train_out = train[['user_idx', 'movie_idx', 'rating', 'timestamp']].copy()
train_out.to_parquet(os.path.join(PROCESSED_DATA_DIR, "train.parquet"), index=False)

test_out = test[['user_idx', 'movie_idx', 'rating', 'timestamp']].copy()
test_out.to_parquet(os.path.join(PROCESSED_DATA_DIR, "test.parquet"), index=False)

movies_out = movies_clean[['movieId', 'movie_idx', 'title', 'genres_clean', 'genome_tag_string', 
                           'raw_tag_string', 'content_string', 'tmdbId', 'imdbId']].copy()
movies_out.to_parquet(os.path.join(PROCESSED_DATA_DIR, "movies_clean.parquet"), index=False)

content_out = movies_clean[['movie_idx', 'content_string']].copy()
content_out.to_csv(os.path.join(PROCESSED_DATA_DIR, "movie_content.csv"), index=False)

constants = {'NUM_USERS': NUM_USERS, 'NUM_MOVIES': NUM_MOVIES}
with open(os.path.join(PROCESSED_DATA_DIR, "dataset_constants.pkl"), "wb") as f:
    pickle.dump(constants, f)


Saving processed files.......



### Verification Assertions

In [ ]:
print("=" * 75)
print("Verification of saved files:")
print("=" * 75)

train_check = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "train.parquet"))
test_check = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "test.parquet"))
movies_check = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "movies_clean.parquet"))


# 1. No timestamp leakage
assert train_check['timestamp'].max() <= test_check['timestamp'].min()
print("PASS 1 — No timestamp leakage between train and test")

# 2. Indices are 0-indexed
assert train_check['user_idx'].min()  == 0
assert train_check['movie_idx'].min() == 0
print("PASS 2 — Indices correctly start at 0")

# 3. Index ranges match encoder sizes
assert train_check['user_idx'].max()  == NUM_USERS  - 1
assert train_check['movie_idx'].max() == NUM_MOVIES - 1
print("PASS 3 — Index max values match NUM_USERS and NUM_MOVIES")

# 4. No nulls anywhere
assert train_check.isnull().sum().sum()   == 0
assert test_check.isnull().sum().sum()    == 0
assert movies_check['content_string'].isnull().sum() == 0
print("PASS 4 — Zero nulls in train, test, and content strings")

# 5. Test users are subset of train users
assert set(test_check['user_idx']).issubset(
       set(train_check['user_idx']))
print("PASS 5 — All test users exist in train (no cold start users)")

# 6. All movie_idxs in test exist in train
assert set(test_check['movie_idx']).issubset(
       set(movies_check['movie_idx']))
print("PASS 6 — All test movies have content strings")

# 7. Content strings not empty
assert (movies_check['content_string'].str.len() > 0).all()
print("PASS 7 — All content strings are non-empty")

print()
print("All 7 assertions passed.")

Verification of saved files:
PASS 1 — No timestamp leakage between train and test
PASS 2 — Indices correctly start at 0
PASS 3 — Index max values match NUM_USERS and NUM_MOVIES
PASS 4 — Zero nulls in train, test, and content strings
PASS 5 — All test users exist in train (no cold start users)
PASS 6 — All test movies have content strings
PASS 7 — All content strings are non-empty

All 7 assertions passed.


In [ ]:
### Final summary
print("=" * 55)
print("PREPROCESSING COMPLETE")
print("=" * 55)
print()
print("Dataset Stats")
print(f"  Train ratings  : {len(train_check):,}")
print(f"  Test ratings   : {len(test_check):,}")
print(f"  NUM_USERS      : {NUM_USERS:,}")
print(f"  NUM_MOVIES     : {NUM_MOVIES:,}")
print()
print("Files Saved")
files = [
    ('train.parquet',             'Training ratings'),
    ('test.parquet',              'Test ratings'),
    ('movies_clean.parquet',      'Clean movie metadata'),
    ('content_strings.csv',       'DistilBERT input strings'),
    ('user_positive_sets.pkl',    'BPR positive sets'),
    ('dataset_constants.pkl',     'NUM_USERS, NUM_MOVIES'),
    ('encoders/user2idx.pkl',     'User ID encoder'),
    ('encoders/movie2idx.pkl',    'Movie ID encoder'),
    ('encoders/idx2user.pkl',     'User ID decoder'),
    ('encoders/idx2movie.pkl',    'Movie ID decoder'),
]
for fname, desc in files:
    path = os.path.join(PROCESSED_DATA_DIR, fname)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024**2
        print(f"  {fname:<35s} {size:.1f} MB  ← {desc}")
print()
print("What each downstream file uses:")
print("  dataset.py    ← train.parquet, user_positive_sets.pkl")
print("  embeddings.py ← content_strings.csv")
print("  chromadb.py   ← embeddings output")
print("  evaluate.py   ← test.parquet")
print("  recommend.py  ← idx2movie.pkl, movies_clean.parquet")
print("  db/crud.py    ← movies_clean.parquet → SQLite")
print()

PREPROCESSING COMPLETE

Dataset Stats
  Train ratings  : 25,868,311
  Test ratings   : 795,563
  NUM_USERS      : 173,134
  NUM_MOVIES     : 27,766

Files Saved
  train.parquet                       177.9 MB  ← Training ratings
  test.parquet                        6.2 MB  ← Test ratings
  movies_clean.parquet                10.4 MB  ← Clean movie metadata
  user_positive_sets.pkl              46.5 MB  ← BPR positive sets
  dataset_constants.pkl               0.0 MB  ← NUM_USERS, NUM_MOVIES
  encoders/user2idx.pkl               3.2 MB  ← User ID encoder
  encoders/movie2idx.pkl              0.5 MB  ← Movie ID encoder
  encoders/idx2user.pkl               3.2 MB  ← User ID decoder
  encoders/idx2movie.pkl              0.5 MB  ← Movie ID decoder

What each downstream file uses:
  dataset.py    ← train.parquet, user_positive_sets.pkl
  embeddings.py ← content_strings.csv
  chromadb.py   ← embeddings output
  evaluate.py   ← test.parquet
  recommend.py  ← idx2movie.pkl, movies_clean.parque